In [1]:
import io
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Inline data provided in the assessment
raw_data = """CustomerID,Annual_Income,Spending_Score,Age
1,15,39,25
2,15,81,35
3,16,6,43
4,16,77,52
5,17,40,35
6,17,76,43
7,70,10,36
8,71,35,55
9,72,5,41
10,87,92,32
11,88,97,30
12,137,18,47"""

def find_best_k_and_summarize(raw_data):
    # Step 1: Load data
    df = pd.read_csv(io.StringIO(raw_data))
    
    # Select features for clustering (Excluding CustomerID)
    features = ['Annual_Income', 'Spending_Score', 'Age']
    X = df[features]
    
    # Step 2: Scale features using StandardScaler
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Step 3: Test K from 2 to 6 inclusive, find highest silhouette score
    best_k = 2
    best_score = -1
    
    print("Silhouette Scores for each K:")
    for k in range(2, 7):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
        labels = kmeans.fit_predict(X_scaled)
        
        score = silhouette_score(X_scaled, labels)
        print(f"K={k} ~ {score:.2f}")
        
        if score > best_score:
            best_score = score
            best_k = k
            
    print(f"\nBest K selected: {best_k}")
    
    # Step 4 & 5: Fit final model with best K and add cluster labels to dataframe
    final_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init='auto')
    df['Cluster'] = final_kmeans.fit_predict(X_scaled)
    
    # Print the cluster summary (mean of each feature grouped by cluster)
    print("\nFinal Cluster Summary (Mean values):")
    # Drop CustomerID before taking mean for cleaner output
    summary = df.groupby('Cluster')[features].mean()
    print(summary)

if __name__ == "__main__":
    find_best_k_and_summarize(raw_data)

Silhouette Scores for each K:
K=2 ~ 0.29
K=3 ~ 0.20
K=4 ~ 0.34
K=5 ~ 0.32
K=6 ~ 0.29

Best K selected: 4

Final Cluster Summary (Mean values):
         Annual_Income  Spending_Score        Age
Cluster                                          
0            29.500000       23.750000  34.750000
1            93.333333       19.333333  47.666667
2            16.000000       78.000000  43.333333
3            87.500000       94.500000  31.000000
